In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub
import math
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from sklearn.linear_model import Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor

from catboost import CatBoostRegressor

from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
'''
import warnings
warnings.filterwarnings('ignore')
'''

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
import pandas as pd

# Load the dataset
credit_path = os.path.join(path, 'Q3_data.csv')                                  # Builds the full file path: downloaded folder + vehicles.csv.

credit_df = pd.read_csv(credit_path)

print( "Sahpe: ", credit_df.shape)


In [ ]:
# Task 2: Write your code here:
credit_df.head()

In [ ]:
# Task 3: Write your code here:

credit_df.info()

In [ ]:
# Task 4: Write your code here:
credit_df.describe()

In [ ]:
# Task 1: Write your code here:

# First we Check Missing

# Is there any missing data?
has_missing = credit_df.isnull().values.any()
print("Is There Any Missing Data?")
print("\t- ", has_missing)

# Total missing values in the whole dataset
total_missing = credit_df.isnull().sum().sum()
print("\nTotal Missing Values in Dataset:")
print("\t- ", total_missing)

# Missing values per column
missing_per_col = credit_df.isnull().sum()
print("\nMissing Values per Column:")
print(missing_per_col)





# Handle them
print('-'*70)
print("Drop Missing")
print("Original shape:", credit_df.shape)

# Drop Coulmns Since data is high dimention
credit_df = credit_df.dropna(axis=1)

print("New shape:", credit_df.shape)

print("\nIs There Any Missing Data After Dropping Columns?")
print("\t- ", credit_df.isnull().values.any())





In [ ]:
# Task 2: Write your code here:

# Function: Check & Drop Duplicates
def check_duplicates(df):

  duplicates = df.duplicated().sum()                                            # .duplicated(): returns True/False for each row, True if the row is a duplicate of a previous row  -  .sum(): counts how many True values → total number of duplicates
  print(f"Number of Duplicate Samples: {duplicates}")

  if duplicates > 0:                                                            # Checks if duplicates exist (more than 0) ?  If yes, Remove them
    print("\n- Dropping Duplicates...")
    df.drop_duplicates(inplace=True)                                            # inplace=True: the original df is modified directly (no new copy) ; permanently removes duplicates from df
    print("\n*** Duplicates Dropped. ***")

  else:
    print("\n- No Duplicate Samples Found.")


# Fun Call
print("Before Droping Duplicate shape:", credit_df.shape)
check_duplicates(credit_df)
print("\n\nAfter Droping Duplicate shape:", credit_df.shape)

In [ ]:

# Task 3: Write your code here:

# Do we have categorical columns?
categorical_cols = credit_df.select_dtypes(include=["object"]).columns
print("Categorical Columns:", list(categorical_cols))
print("Number of Categorical Columns:", len(list(categorical_cols)))




In [ ]:
# Debug
print(f"\nBefore Scalling ranges ")
credit_df.head()

In [ ]:
# Task 4: Write your code here:
# Apply feature scaling for all features (Use StandardScaler)


# Select numeric features
numerical_cols = credit_df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET
# column counts
print("Number of Numeric features:", len(credit_df.columns))


# Scale
# Instantiate StandardScaler
standard_scaler = StandardScaler()

# fit_trnasform()
credit_df[numerical_cols] = standard_scaler.fit_transform(credit_df[numerical_cols])


# Debug
print(f"\nAfter Scalling ranges ")
credit_df.head()

In [ ]:
# Task 5: Write your code here:
#  What does our target variable (charges) look like?
# Function: Check if target imbalanced?
import seaborn as sns

def check_target_imbalance(df, target_column):
  print("Target Distribution: \n\t")
  print(df[target_column].value_counts(normalize=True))                         # prints class distribution as percentages: df[target_column]: selects the target column  -  .value_counts(): counts the number of occurrences for each class  -  normalize=True: converts counts into proportions (percentages).

  # Plot
  sns.countplot(x=df[target_column])                                            # Bar chart showing the count of each class -   x-axis: target classes (0 and 1)  -   y-axis: how many samples belong to each class
  plt.title("Target Distribution")
  plt.show()



check_target_imbalance(credit_df, "Target")

print("Yes it's imbalance")

In [ ]:
# Task 1: Write your code here:
# Separate Features and Target

target_column = "Target"


X = credit_df.drop(target_column, axis=1)
y = credit_df[target_column]



# Debug
print("\nDataset Shape Before Splitting:" , credit_df.shape)
print("Features X: \n\t", list(X.columns) )
print("\nNumber of Features X: \n\t", X.shape[1] )
print("\nShape of Features X: \n\t", X.shape )
print("")
print("-"*70)
print("")
print("\nTrget Label y: \n\t", y.name )
print("\nNumber of Trget Values: \n\t", y.shape[0] )
print("\nShape of Trget Label y: \n\t", y.shape )
print(y)

In [ ]:
credit_df.head(10)

In [ ]:
import catboost

In [ ]:
# Task 2,3,4,5: Write your code here:

# Create K-Fold Cross Validation
n_folds = 5                                                                     # K=5 Folds

skf = StratifiedKFold(
    n_splits= n_folds,                                                          # split data into 5 folds
    shuffle= True,                                                              # shuffle data before splitting
    random_state= 42                                                            # reproducible splits
)


# Train RandomForestClassifier
model = CatBoostRegressor(verbose=0  )


# Stratified K-Fold Training Loop
# Storage for RF results for each fold
Acc_scores = []
F1_scores = []



for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):          # Uses X and y, Ensures class balance in each fold, Returns indices, not data
  print(f"\nFold {fold_idx + 1}/{n_folds}")

  # Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]                     #    # Select training and testing data using indices
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  # train
  model.fit(X_train, y_train)

  # predict
  y_pred = model.predict(X_test) # validate

  print( y_pred)
  # Evaluate - Save metrics for that model in this fold
  accuracy = accuracy_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred, zero_division=0)

  Acc_scores.append(accuracy)
  F1_scores.append(f1)




# Final average scores across  of each evaluation metric across folds
print("-"*70)
print("\nAverage across folds")
print(f"  Accuracy : {np.mean(Acc_scores):.2f}")
print(f"  F1-Score : {np.mean(F1_scores):.2f}")

In [ ]:
y_pred

In [ ]:
# Task 1: Write your code here:
# Stores the feature names (column names of X)
feature_cols = X.columns                                                            # will be used as labels in the plot
print(feature_cols)

# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
golden = ' '
print('most important feature.. the golden feature:' , golden)

In [ ]:
# Task Bonus: Write your code here: